# 04 — Policy sensitivity: tau, tau_review, theta_mule and the zone rule

Reads `artifacts/engine_tune.json` (from `python -m engine.tune`, v2 held-out split) and `artifacts/sim/pareto.json`
(from `python -m sim.run pareto`, independent suite). Nothing here is typed in: every number is read from those files.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ART = Path(os.environ.get("NYAYA_ARTIFACTS", Path.home() / "shared" / "nyaya" / "artifacts"))
pd.set_option("display.width", 200)

In [ ]:
tune = json.loads((ART / "engine_tune.json").read_text())
wide = pd.DataFrame(tune["grid_wide"])
wide["leak_pct"] = 100 * wide.fraud_leakage_inr / wide.fraud_total_inr
print("rows scored:", tune["rows_scored"])
cols = ["tau", "tau_review", "theta_mule", "zone_hold_needs_mule", "victims_left_zero_rate", "leak_pct",
        "legit_review_rate", "collector_donation_hold", "syndicate_hold"]
wide[wide.tau_review == 0][cols].round(4)

## The trade-off the zone rule makes

Each point is one (tau, theta_mule) setting with tau_review = 0. Left: victims left with Rs 0. Right: share of fraud
rupees that leak. Orange = zone rule on (only a mule flag can freeze inside a disaster zone).

In [ ]:
sub = wide[wide.tau_review == 0]
fig, ax = plt.subplots(figsize=(7, 4.5))
for zone, color in [(False, "tab:blue"), (True, "tab:orange")]:
    s = sub[sub.zone_hold_needs_mule == zone]
    ax.scatter(100 * s.victims_left_zero_rate, s.leak_pct, c=color, label=f"zone rule {'on' if zone else 'off'}")
chosen = tune["policy_file"]
ax.set_xlabel("victims left with Rs 0 (% of displaced payments)")
ax.set_ylabel("fraud leakage (% of fraud rupees)")
ax.set_title(f"Policy grid, v2 held-out split ({tune['rows_scored']} rows)")
ax.legend(); plt.tight_layout(); plt.show()
print("policy in force:", {k: chosen.get(k) for k in ["tau", "tau_review", "theta_mule", "zone_hold_needs_mule"]})

## Pareto sweep on the independent suite (`sim.run pareto`)

One parameter moves at a time around the policy in force. `loses_to` lists any baseline with lower total expected harm
at that setting — the regions where NYAYA does worse are shown, not hidden.

In [ ]:
par = json.loads((ART / "sim" / "pareto.json").read_text())
print("suite:", par["suite_rows"], "rows from", par.get("suite_source"))
rows = [{"parameter": p["parameter"], "value": p["value"],
         "victims_zero_rate": p["metrics"]["victims_left_with_zero_rate"],
         "fraud_leakage_rs": p["metrics"]["fraud_leakage_rupees"],
         "expected_harm_rs": p["metrics"]["total_expected_harm_rupees"],
         "loses_to": ", ".join(p["loses_to"]) or "-"} for p in par["points"]]
pts = pd.DataFrame(rows)
base = {k: v["total_expected_harm_rupees"] for k, v in par["baselines"].items()}
print("baseline expected harm (Rs):", {k: round(v) for k, v in base.items()})
pts.round(4)

In [ ]:
fig, axes = plt.subplots(1, len(par["grids"]), figsize=(4 * len(par["grids"]), 3.5), sharey=True)
for ax, (name, g) in zip(axes, pts.groupby("parameter")):
    ax.plot(g["value"], g["expected_harm_rs"] / 1e5, marker="o", label="NYAYA")
    for b, v in base.items():
        ax.axhline(v / 1e5, ls="--", lw=1, label=b)
    ax.set_title(name); ax.set_xlabel("value")
axes[0].set_ylabel("total expected harm (Rs lakh)"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()